# Inspeção inicial de um snapshot do CNPJ

Este notebook verifica se um snapshot mensal do CNPJ foi organizado corretamente no padrão do projeto e coleta amostras úteis para validação.

## Objetivos

- conferir a estrutura das pastas do snapshot
- listar famílias e arquivos extraídos
- amostrar linhas reais dos arquivos
- gerar um resumo inicial para documentação do mês

In [ ]:
from pathlib import Path
import json

RAIZ_DADOS = Path('/content/dados_brutos/cnpj')
SNAPSHOT_MES = '2026-04'
SNAPSHOT = RAIZ_DADOS / SNAPSHOT_MES

SNAPSHOT

In [ ]:
partes = [
    '00_pacote_original',
    '01_subarquivos_zip',
    '02_extraido_texto',
    '03_processado',
    '04_metadados',
]

estrutura = {parte: (SNAPSHOT / parte).exists() for parte in partes}
estrutura

In [ ]:
familias = {
    'empresas': SNAPSHOT / '02_extraido_texto' / 'empresas',
    'estabelecimentos': SNAPSHOT / '02_extraido_texto' / 'estabelecimentos',
    'simples': SNAPSHOT / '02_extraido_texto' / 'simples',
    'socios': SNAPSHOT / '02_extraido_texto' / 'socios',
    'dominios': SNAPSHOT / '02_extraido_texto' / 'dominios',
}

resumo_familias = {}
for familia, pasta in familias.items():
    arquivos = sorted([arquivo for arquivo in pasta.iterdir() if arquivo.is_file()]) if pasta.exists() else []
    resumo_familias[familia] = {
        'quantidade_arquivos': len(arquivos),
        'primeiros_arquivos': [arquivo.name for arquivo in arquivos[:3]],
        'bytes_total': sum(arquivo.stat().st_size for arquivo in arquivos),
    }

resumo_familias

In [ ]:
def amostrar_linhas(caminho: Path, quantidade: int = 3):
    linhas = []
    with caminho.open('r', encoding='latin-1') as arquivo:
        for _ in range(quantidade):
            linha = arquivo.readline()
            if not linha:
                break
            linhas.append(linha.strip())
    return linhas

arquivos_amostra = {
    'cnae': SNAPSHOT / '02_extraido_texto' / 'dominios' / 'F.K03200$Z.D60411.CNAECSV',
    'municipios': SNAPSHOT / '02_extraido_texto' / 'dominios' / 'F.K03200$Z.D60411.MUNICCSV',
    'empresas': next((SNAPSHOT / '02_extraido_texto' / 'empresas').iterdir()),
    'estabelecimentos': next((SNAPSHOT / '02_extraido_texto' / 'estabelecimentos').iterdir()),
    'simples': next((SNAPSHOT / '02_extraido_texto' / 'simples').iterdir()),
}

{chave: amostrar_linhas(caminho) for chave, caminho in arquivos_amostra.items()}

In [ ]:
manifesto = {
    'snapshot_mes': SNAPSHOT_MES,
    'estrutura': estrutura,
    'resumo_familias': resumo_familias,
}

saida = SNAPSHOT / '04_metadados' / 'manifest_colab.json'
saida.write_text(json.dumps(manifesto, ensure_ascii=False, indent=2), encoding='utf-8')
saida